<a href="https://colab.research.google.com/github/M-Abbi/Probability-Statistics-Bootcamp/blob/main/Variance%2C_Covariance_%26_Correlation_Computation_Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Foundational Risk Architecture: Variance, Covariance, and Correlation

In quantitative finance, we don't look at assets in isolation. Managing risk requires understanding not only how much an individual asset's returns fluctuate, but also how those fluctuations interact with every other asset in the portfolio.

---

## 1. Standalone Risk: Variance ($\text{Var}$)

By definition, **Variance** is the expected value of the squared deviations from the mean. It quantifies the absolute dispersion or "spread" of an asset's return distribution.

$$\text{Var}(X) = \sigma_x^2 = \mathbb{E}\left[ (X - \mu_x)^2 \right]$$

Using the linearity of expectation, this expands into a highly practical computational form:
$$\text{Var}(X) = \mathbb{E}[X^2] - (\mathbb{E}[X])^2$$

* **Front-Desk Reality:** While variance sets the absolute mathematical scale of risk, its units are squared (e.g., $\%\text{ squared}$), which lacks intuitive meaning. Quants take the square root to calculate **Standard Deviation ($\sigma_x$)**, commonly referred to as **Volatility**.

---

## 2. Joint Risk: Covariance ($\text{Cov}$)

When combining assets, we must evaluate their co-movement. **Covariance** measures the directional relationship between the returns of two random variables. It tells us whether two assets tend to rally together, move in opposition, or behave completely independently.

$$\text{Cov}(X,Y) = \sigma_{xy} = \mathbb{E}\left[ (X - \mu_x)(Y - \mu_y) \right]$$

Expanding this via expectation algebra yields:
$$\text{Cov}(X,Y) = \mathbb{E}[XY] - \mathbb{E}[X]\mathbb{E}[Y]$$



[Image of scatter plots showing positive, negative, and zero covariance]


### Interpreting the Sign of Covariance:
* **$\text{Cov}(X,Y) > 0$:** The assets move in the same direction. When asset $X$ outperforms its mean, asset $Y$ typically does too.
* **$\text{Cov}(X,Y) < 0$:** The assets move in opposite directions, creating a natural structural hedge.
* **$\text{Cov}(X,Y) = 0$:** The asset movements are linearly independent (orthogonal).

---

## 3. Standardizing Co-Movement: Correlation ($\rho$)

While covariance dictates the direction of co-movement, its absolute value is highly scale-dependent. For instance, changing an asset's quote metric from Dollars to Cents scales the covariance by $10,000$, even though the economic dependency hasn't altered at all.

To eliminate scale dependence and create a pure metric of linear dependency, the front desk standardizes covariance by dividing it by the product of the assets' individual standard deviations. This yields the **Pearson Correlation Coefficient ($\rho$)**:

$$\rho_{xy} = \frac{\text{Cov}(X,Y)}{\sigma_x \sigma_y} \quad \implies \quad \text{Cov}(X,Y) = \rho_{xy} \sigma_x \sigma_y$$

By the Cauchy-Schwarz inequality, correlation is strictly bounded between $-1$ and $+1$:
$$\rho_{xy} \in [-1, 1]$$

| Correlation Value | Relationship Profile | Impact on Portfolio Risk ($\sigma_P$) |
| :--- | :--- | :--- |
| **$\rho = +1.0$** | Perfect Positive Linear Dependency | No diversification benefit. Risk scales linearly. |
| **$\rho = 0.0$** | Completely Uncorrelated | Diversification occurs naturally through mathematical orthogonality. |
| **$\rho = -1.0$** | Perfect Negative Linear Dependency | A mathematically risk-free portfolio ($\sigma_P = 0$) can be perfectly engineered. |

---

## 4. The Multi-Asset Scaling Engine: The Covariance Matrix ($\mathbf{\Sigma}$)

On an institutional trading desk managing an $N$-asset portfolio, writing scalar algebraic equations for every pairwise interaction becomes completely intractable. Instead, we arrange every individual variance and pairwise covariance parameter cleanly into an $N \times N$ linear algebra framework known as the **Covariance Matrix ($\mathbf{\Sigma}$)**:

$$\mathbf{\Sigma} = \begin{bmatrix}
\text{Var}(R_1) & \text{Cov}(R_1, R_2) & \dots & \text{Cov}(R_1, R_N) \\
\text{Cov}(R_2, R_1) & \text{Var}(R_2) & \dots & \text{Cov}(R_2, R_N) \\
\vdots & \vdots & \ddots & \vdots \\
\text{Cov}(R_N, R_1) & \text{Cov}(R_N, R_2) & \dots & \text{Var}(R_N)
\end{bmatrix} = \begin{bmatrix}
\sigma_1^2 & \rho_{12}\sigma_1\sigma_2 & \dots & \rho_{1N}\sigma_1\sigma_N \\
\rho_{21}\sigma_2\sigma_1 & \sigma_2^2 & \dots & \rho_{2N}\sigma_2\sigma_N \\
\vdots & \vdots & \ddots & \vdots \\
\rho_{N1}\sigma_N\sigma_1 & \rho_{N2}\sigma_N\sigma_2 & \dots & \sigma_N^2
\end{bmatrix}$$



### The Quadratic Form of Portfolio Variance
Let $\mathbf{w} = [w_1, w_2, \dots, w_N]^T$ represent a vector containing the portfolio allocation weights assigned to each asset. Using matrix multiplication, the total portfolio variance compresses down into a singular, clean quadratic expression:

$$\text{Var}(P) = \mathbf{w}^T \mathbf{\Sigma} \mathbf{w}$$

To calculate the overall annualized portfolio volatility ($\sigma_P$), the desk takes the square root:
$$\sigma_P = \sqrt{\mathbf{w}^T \mathbf{\Sigma} \mathbf{w}}$$

In [1]:
import numpy as np

def run_master_risk_simulation():
    """
    A comprehensive numerical simulation that computes raw variance, covariance,
    correlation, and the final multi-asset covariance matrix from sample historical data paths.
    """
    np.random.seed(101)
    n_days = 252 * 5  # Simulating 5 years of daily trading data

    # 1. Generate Raw Correlated Asset Returns (The Underlying Chaos)
    # Let's simulate distinct market behaviors
    market_factor = np.random.normal(0, 0.01, n_days)  # Systemic market shock
    gold_factor = np.random.normal(0, 0.008, n_days)   # Safe-haven flight factor

    returns_tech = 0.6 * market_factor + np.random.normal(0, 0.015, n_days)   # Highly sensitive to market
    returns_bonds = -0.2 * market_factor + np.random.normal(0, 0.004, n_days) # Negative market correlation
    returns_gold = -0.4 * market_factor + gold_factor                        # Flight to safety vehicle

    # Pack returns together for easy processing
    raw_data = np.vstack([returns_tech, returns_bonds, returns_gold])
    asset_names = ['Tech (X)', 'Bonds (Y)', 'Gold (Z)']

    print("=========================================================")
    print("        PHASE 1: EXTRACTING SCALAR STANDALONE RISK        ")
    print("=========================================================")
    means = [np.mean(r) for r in raw_data]
    variances = [np.var(r) for r in raw_data]
    volatilities = [np.sqrt(v) for v in variances]

    for i, name in enumerate(asset_names):
        print(f"{name:<10} | Daily Mean: {means[i]:.6f} | Daily Var: {variances[i]:.6f} | Daily Vol: {volatilities[i]*100:.2f}%")
    print("=========================================================\n")

    print("=========================================================")
    print("        PHASE 2: DECONSTRUCTING CO-MOVEMENT MATRIX       ")
    print("=========================================================")
    # Let's manually calculate Covariance and Correlation between Tech (X) and Gold (Z)
    X = returns_tech
    Z = returns_gold

    manual_cov_xz = np.mean((X - np.mean(X)) * (Z - np.mean(Z)))
    manual_rho_xz = manual_cov_xz / (np.std(X) * np.std(Z))

    print(f"Manual Covariance (Tech, Gold) : {manual_cov_xz:.6f}")
    print(f"Manual Correlation (Tech, Gold): {manual_rho_xz:.4f}  <-- Clean, scale-free dependency indicator")
    print("=========================================================\n")

    print("=========================================================")
    print("        PHASE 3: CONSTRUCTING THE MATRIX ENGINE (Sigma)  ")
    print("=========================================================")
    # Compute the full empirical Covariance Matrix using numpy's matrix operations
    Sigma = np.cov(raw_data)

    print("Empirical Covariance Matrix (Σ):")
    print("            [ Tech (X)  ] [ Bonds (Y) ] [ Gold (Z)  ]")
    for i, name in enumerate(['Tech ', 'Bonds', 'Gold ']):
        print(f"{name} | " + " ".join(f"{val:11.7f}" for val in Sigma[i]))
    print("=========================================================\n")

    print("=========================================================")
    print("        PHASE 4: VECTORS-MATRIX PORTFOLIO VAILDATION     ")
    print("=========================================================")
    # Define an asset allocation strategy (Must sum to 1.0)
    w = np.array([0.40, 0.40, 0.20]) # 40% Tech, 40% Bonds, 20% Gold

    # Execute the Quadratic Matrix Form: Var(P) = w^T * Sigma * w
    portfolio_variance = w.T @ Sigma @ w
    portfolio_volatility_daily = np.sqrt(portfolio_variance)
    portfolio_volatility_annualized = portfolio_volatility_daily * np.sqrt(252) # Scale risk to annual horizon

    # Naive calculations ignoring covariance entirely
    naive_variance = np.sum((w**2) * variances)
    naive_volatility_annualized = np.sqrt(naive_variance) * np.sqrt(252)

    print(f"Allocation Vector (w)                : {w}")
    print(f"True Annualized Portfolio Volatility : {portfolio_volatility_annualized*100:.2f}%")
    print(f"Naive Annual Vol (Ignoring Covariance): {naive_volatility_annualized*100:.2f}%")
    print(f"Diversification Risk Alpha Benefit   : {(naive_volatility_annualized - portfolio_volatility_annualized)*100:.2f}%")
    print("=========================================================")

run_master_risk_simulation()

        PHASE 1: EXTRACTING SCALAR STANDALONE RISK        
Tech (X)   | Daily Mean: 0.000377 | Daily Var: 0.000270 | Daily Vol: 1.64%
Bonds (Y)  | Daily Mean: -0.000022 | Daily Var: 0.000021 | Daily Vol: 0.46%
Gold (Z)   | Daily Mean: 0.000393 | Daily Var: 0.000082 | Daily Vol: 0.91%

        PHASE 2: DECONSTRUCTING CO-MOVEMENT MATRIX       
Manual Covariance (Tech, Gold) : -0.000015
Manual Correlation (Tech, Gold): -0.1013  <-- Clean, scale-free dependency indicator

        PHASE 3: CONSTRUCTING THE MATRIX ENGINE (Sigma)  
Empirical Covariance Matrix (Σ):
            [ Tech (X)  ] [ Bonds (Y) ] [ Gold (Z)  ]
Tech  |   0.0002705  -0.0000105  -0.0000151
Bonds |  -0.0000105   0.0000214   0.0000097
Gold  |  -0.0000151   0.0000097   0.0000822

        PHASE 4: VECTORS-MATRIX PORTFOLIO VAILDATION     
Allocation Vector (w)                : [0.4 0.4 0.2]
True Annualized Portfolio Volatility : 10.74%
Naive Annual Vol (Ignoring Covariance): 11.22%
Diversification Risk Alpha Benefit   : 0.48%
